In [1]:
import pandas as pd
import numpy as np

In [2]:
# ===============================
# 1. LOAD DATA
# ===============================

df = pd.read_csv("data_2.1.csv")


In [3]:
# переконаємося, що дати - у форматі datetime
df['install_date'] = pd.to_datetime(df['install_date'])
df['event_date'] = pd.to_datetime(df['event_date'])

In [4]:
# ===============================
# 2. CLEAN DATA 
# ===============================

# прибираємо refund
df_clean = df[df['refund'] != 1]


In [5]:
# прибираємо event_date < install_date
df_clean = df_clean[df_clean['event_date'] >= df_clean['install_date']]

In [6]:
print("Users after cleaning:", df_clean['user_id'].nunique())

Users after cleaning: 3240


In [7]:
# ===============================
# 3. Залишаємо тільки TRIAL USERS
# ===============================

trial_users = (
    df_clean[df_clean['payment_type'] == 'free_trial']['user_id']
    .unique()
)

df_trials = df_clean[df_clean['user_id'].isin(trial_users)]

print("Trial users:", len(trial_users))

Trial users: 3187


In [8]:
# ===============================
# 4. Рахуємо recurrent_count (як CTE user_payments)
# ===============================

recurrent_counts = (
    df_trials[df_trials['payment_type'] == 'recurrent']
    .groupby('user_id')
    .size()
    .reset_index(name='recurrent_count')
)

In [9]:
# додаємо користувачів без оплат
all_trials = pd.DataFrame({'user_id': trial_users})

user_payments = all_trials.merge(
    recurrent_counts,
    on='user_id',
    how='left'
)

user_payments['recurrent_count'] = user_payments['recurrent_count'].fillna(0)

In [10]:
# ===============================
# 5. BUILD FUNNEL (GROSS TOTALS)
# ===============================

trials = len(user_payments)

first  = (user_payments['recurrent_count'] >= 1).sum()
second = (user_payments['recurrent_count'] >= 2).sum()
third  = (user_payments['recurrent_count'] >= 3).sum()
fourth = (user_payments['recurrent_count'] >= 4).sum()
fifth  = (user_payments['recurrent_count'] >= 5).sum()

print("\n=== FUNNEL COUNTS ===")
print("Trials:", trials)
print("1st payment:", first)
print("2nd payment:", second)
print("3rd payment:", third)
print("4th payment:", fourth)
print("5th payment:", fifth)



=== FUNNEL COUNTS ===
Trials: 3187
1st payment: 1117
2nd payment: 831
3rd payment: 690
4th payment: 591
5th payment: 524


In [11]:
# ===============================
# 6. CONDITIONAL CONVERSION RATES
# ===============================

CR1 = first / trials if trials > 0 else 0
CR2 = second / first if first > 0 else 0
CR3 = third / second if second > 0 else 0
CR4 = fourth / third if third > 0 else 0
CR5 = fifth / fourth if fourth > 0 else 0

print("\n=== CONDITIONAL CR ===")
print("CR1 (Trial → 1):", round(CR1, 4))
print("CR2 (1 → 2):", round(CR2, 4))
print("CR3 (2 → 3):", round(CR3, 4))
print("CR4 (3 → 4):", round(CR4, 4))
print("CR5 (4 → 5):", round(CR5, 4))


=== CONDITIONAL CR ===
CR1 (Trial → 1): 0.3505
CR2 (1 → 2): 0.744
CR3 (2 → 3): 0.8303
CR4 (3 → 4): 0.8565
CR5 (4 → 5): 0.8866


In [12]:
# ===============================
# 7. EXPECTED PAYMENTS (GEOMETRIC MODEL)
# ===============================

expected_payments = CR1 * (
    1 +
    CR2 * (
        1 +
        CR3 * (
            1 +
            CR4 * (
                1 +
                CR5
            )
        )
    )
)

print("\nExpected lifetime payments:", round(expected_payments, 4))


Expected lifetime payments: 1.1776


In [13]:
# ===============================
# 8. LTV CALCULATION
# ===============================

price = 3.49

LTV = price * expected_payments

print("\nExpected LTV:", round(LTV, 4))


Expected LTV: 4.1098


In [14]:
# ===============================
# 9. BUILD SUMMARY TABLE
# ===============================

summary = pd.DataFrame({
    "Metric": [
        "Trials",
        "1st payment",
        "2nd payment",
        "3rd payment",
        "4th payment",
        "5th payment",
        "CR1",
        "CR2",
        "CR3",
        "CR4",
        "CR5",
        "Expected payments",
        "Expected LTV"
    ],
    "Value": [
        trials,
        first,
        second,
        third,
        fourth,
        fifth,
        CR1,
        CR2,
        CR3,
        CR4,
        CR5,
        expected_payments,
        LTV
    ]
})

summary

,Metric,Value
0,Trials,3187.000000
1,1st payment,1117.000000
2,2nd payment,831.000000
3,3rd payment,690.000000
4,4th payment,591.000000
5,5th payment,524.000000
6,CR1,0.350486
7,CR2,0.743957
8,CR3,0.830325
9,CR4,0.856522


In [15]:
# ===============================
# 1. DROP-OFF CALCULATION
# ===============================

drop_1 = trials - first
drop_2 = first - second
drop_3 = second - third
drop_4 = third - fourth
drop_5 = fourth - fifth


In [16]:
# ===============================
# 2. SANKEY EDGES
# ===============================

sankey_data = pd.DataFrame({

    "source": [
        "Trial",
        "Trial",
        "1st Payment",
        "1st Payment",
        "2nd Payment",
        "2nd Payment",
        "3rd Payment",
        "3rd Payment",
        "4th Payment",
        "4th Payment"
    ],

    "target": [
        "1st Payment",
        "Churn after Trial",
        "2nd Payment",
        "Churn after 1st",
        "3rd Payment",
        "Churn after 2nd",
        "4th Payment",
        "Churn after 3rd",
        "5th Payment",
        "Churn after 4th"
    ],

    "value": [
        first,
        drop_1,
        second,
        drop_2,
        third,
        drop_3,
        fourth,
        drop_4,
        fifth,
        drop_5
    ]
})

sankey_data

,source,target,value
0,Trial,1st Payment,1117
1,Trial,Churn after Trial,2070
2,1st Payment,2nd Payment,831
3,1st Payment,Churn after 1st,286
4,2nd Payment,3rd Payment,690
5,2nd Payment,Churn after 2nd,141
6,3rd Payment,4th Payment,591
7,3rd Payment,Churn after 3rd,99
8,4th Payment,5th Payment,524
9,4th Payment,Churn after 4th,67


In [17]:
# ===============================
# 3. ADD CONVERSION RATES
# ===============================

sankey_data["conversion_rate"] = [
    CR1,
    1-CR1,
    CR2,
    1-CR2,
    CR3,
    1-CR3,
    CR4,
    1-CR4,
    CR5,
    1-CR5
]

sankey_data

,source,target,value,conversion_rate
0,Trial,1st Payment,1117,0.350486
1,Trial,Churn after Trial,2070,0.649514
2,1st Payment,2nd Payment,831,0.743957
3,1st Payment,Churn after 1st,286,0.256043
4,2nd Payment,3rd Payment,690,0.830325
5,2nd Payment,Churn after 2nd,141,0.169675
6,3rd Payment,4th Payment,591,0.856522
7,3rd Payment,Churn after 3rd,99,0.143478
8,4th Payment,5th Payment,524,0.886633
9,4th Payment,Churn after 4th,67,0.113367


In [18]:
sankey_data.to_csv("sankey_ltv_data.csv", index=False)